<a href="https://colab.research.google.com/github/MusaR10/AAI2025/blob/2026fall/Customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Generate sample customer data
#data = {
 #   'Age': [25, 34, 45, 28, 52, 36, 41, 29, 47, 33],
 #  'Tenure': [10, 50, 20, 15, 60, 30, 25, 12, 55, 40],
 #   'Balance': [100, 250, 150, 80, 300, 200, 175, 90, 280, 220],
 #   'NumOfProducts': [5, 2, 8, 6, 1, 3, 7, 4, 0, 2],
 #   'Geography': ['North', 'South', 'West', 'East', 'South', 'North', 'West', 'East',
 #              'South', 'North'],
  #  'churn': [1, 0, 1, 1, 0, 0, 1, 1, 0, 0] # 1 = churned, 0 = not churned
#}
#df = pd.DataFrame(data)
# got the data from https://github.com/MujtabaInsightCore/Bank-Customer-Churn/blob/main/Customer-Churn-Records.csv
df = pd.read_csv('/content/Customer-Churn-Records.csv')
# Features and target
X = df[['Age', 'Tenure', 'Balance', 'NumOfProducts', 'Satisfaction Score', 'EstimatedSalary',
        'Geography', 'Gender']]
y = df['Exited']

# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age', 'Tenure','EstimatedSalary', 'Balance','Satisfaction Score',
                                   'NumOfProducts']),
        ('cat', OneHotEncoder(sparse_output=False), ['Geography', 'Gender'])
    ])

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    'Age': [35],
    'Tenure': [1],
    'Balance': [515000],
    'NumOfProducts': [2],
    'EstimatedSalary': [68000],
    'Satisfaction Score': [4],
    'Gender': ['Female'],
    'Geography': ['France']
})
churn_probability = model.predict_proba(new_customer)[0][1]
# Probability of churn (class 1)
new_customer2 = pd.DataFrame({
    'Age': [35],
    'Tenure': [1],
    'Balance': [515000],
    'NumOfProducts': [2],
    'EstimatedSalary': [68000],
    'Satisfaction Score': [4],
    'Gender': ['Female'],
    'Geography': ['Germany']
})
churn_probability2 = model.predict_proba(new_customer2)[0][1]
# Probability of churn (class 1)

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0
churn_prediction2 = 1 if churn_probability2 > threshold else 0

print(f"Churn Probability for new customer in France: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

print(f"Churn Probability for same customer in Germany: {churn_probability2:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction2}")

# Display model coefficients
feature_names = (model.named_steps['preprocessor']
                 .named_transformers_['cat']
                 .get_feature_names_out(['Geography', 'Gender'])).tolist() + ['Age', 'Tenure','EstimatedSalary', 'Balance','Satisfaction Score', 'NumOfProducts']
coefficients = model.named_steps['classifier'].coef_[0]

print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

Churn Probability for new customer in France: 0.35
Churn Prediction (1 = churn, 0 = no churn): 0
Churn Probability for same customer in Germany: 0.54
Churn Prediction (1 = churn, 0 = no churn): 1

Model Coefficients:
Geography_France: 0.66
Geography_Germany: -0.02
Geography_Spain: 0.02
Gender_Female: 0.16
Gender_Male: 0.00
Age: -0.07
Tenure: -0.54
EstimatedSalary: 0.23
Balance: -0.47
Satisfaction Score: -0.12
NumOfProducts: -0.66
